# Notebook 10: Sensor Failure & Missing-Modality Robustness Study

**Objective**: Quantify model resilience against noisy, degraded, or failing wearable sensor channels.

### Experimental Scenarios:
1. **Random Channel Dropout**: 10%, 25%, 50%, and 75% of on-body channels randomly masked.
2. **Complete Regional Outages**: Total loss of individual body regions (Trunk, Right Arm, Left Arm, Right Leg, Left Leg).

## 1. Configure Experiment Mode

In [ ]:
EXPERIMENT_MODE = "FAST"

import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
api.setup_project()

## 2. Execute Robustness Evaluation (CPU)

In [ ]:
robustness_df = api.run_robustness(model_type="random_forest")

print("\n" + "=" * 75)
print(" SENSOR FAILURE & MISSING-MODALITY ROBUSTNESS RESULTS")
print("=" * 75)
print(robustness_df[["Failure Scenario", "Dropped Channels", "Accuracy", "Macro F1", "Degradation (Macro F1)"]].to_string(index=False))
print("=" * 75)

## 3. Visualize Failure Degradation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel A: Random Dropout Curve
rand_df = robustness_df[robustness_df["Condition Type"] == "Random Dropout"].copy()
drop_pcts = [0, 10, 25, 50, 75]
axes[0].plot(drop_pcts, rand_df["raw_macro_f1"] * 100, marker="o", linewidth=2.5, color="#1f77b4")
axes[0].set_title("Performance vs. Random Sensor Channel Loss")
axes[0].set_xlabel("Percentage of Randomly Dropped Channels (%)")
axes[0].set_ylabel("Macro F1 Score (%)")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Panel B: Regional Outages
reg_df = robustness_df[robustness_df["Condition Type"] == "Regional Failure"].copy()
sns.barplot(data=reg_df, x="Failure Scenario", y="raw_macro_f1", ax=axes[1], palette="Reds_r")
axes[1].set_title("Degradation Under Complete Regional Outages")
axes[1].set_ylabel("Macro F1 Score")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()